# CHEM 1000 - Spring 2026
Prof. Geoffrey Hutchison, University of Pittsburgh

## Neural Networks, Machine Learning and Large Language Models

By the end of this session, you should be able to:
- Understand the basics of neural network models
- How machine learning works (i.e., training, hyperparameters)
- Understand the basics of "large language models" like ChatGPT, Claude, and Gemini

Some of the code examples in this notebook were, in fact, generated by Claude Code, using model Opus 4.7 "extended thinking" with the prompt:

> For my CHEM 1000 - Mathematics for Chemistry class, I'd like to plan a Jupyter notebook in Python to introduce these students (undergrad chem majors) to the basics of how LLM models work, including a brief intro to neural nets in general, training, etc. in one class period. What are some good code examples?

## What is machine learning anyway? And how do "AI models" work?

Today we'll look at some code in `numpy` to understand how something like ChatGPT or Claude actually works under the hood.

> **A language model is just a function that takes a sequence of words and returns a probability distribution over the next word (or block of words). That function is a (very large) neural network whose parameters are fit through training on a lot of text.**

Conceptually, that's it.

Obviously, I'm simplifying. There are a bunch of topics I'm leaving out, including different kinds of neural net models (e.g., [transformers](https://en.wikipedia.org/wiki/Transformer_(deep_learning)) and [attention](https://en.wikipedia.org/wiki/Attention_Is_All_You_Need), different kinds of training and fine-tuning, "thinking" models, etc.)

Most everything else is engineering (e.g., getting all that text, computers, tuning the model, etc.). Which is not nothing, but at least you'll get an idea how these kinds of models can work for other applications in chemistry, chemistry, and physics.

1. What's a neural network?
2. Training = optimization
3. Applying to language
4. Scaling up, limits & other applications (e.g., chemistry)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # the "seed" ensures we all get the same random sequences

%matplotlib inline
%config InlineBackend.figure_format = 'retina' # high-res images where supported
plt.style.use('./chem1000.mplstyle')

---
## A neuron is just a function

In principle, all of this is just nonlinear curve fitting. If we go back to the 1940s and 1950s, scientists and mathematicians worked to understand how neurons and brains "think." Inspired by models of biological neural networks, they proposed [artificial neural networks](https://en.wikipedia.org/wiki/Neural_network_(machine_learning)) that could be trained to approximate essentially any response function.

A single **artificial neuron** takes a vector of inputs $\mathbf{x}$, computes a weighted dot product, adds a bias, and "squashes" the result through a nonlinear **activation function** $\sigma$:

$$y = \sigma\!\left(\mathbf{w}\cdot\mathbf{x} + b\right) = \sigma\!\left(\sum_{i} w_i x_i + b\right)$$

The $w_i$ and $b$ are **parameters** for the model — numbers we'll learn from data. The activation function $\sigma$ is just some fixed nonlinear function. A common choice is the **sigmoid**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

which maps any real number into $(0, 1)$.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

z = np.linspace(-6, 6, 200)
plt.plot(z, sigmoid(z))
plt.axhline(0, color='gray', lw=0.5); plt.axhline(1, color='gray', lw=0.5, ls='--')
plt.xlabel('z'); plt.ylabel(r'$\sigma(z)$'); plt.title('Sigmoid activation')
plt.grid(alpha=0.3); plt.show()

**Conceptual Questions for Discussion**

  1. Our activation function scales from 0 to 1. Can you think of a reason we might want that, rather than an additional parameter to span any real number range?
  2. Why might we want the activation function to be nonlinear like the sigmoid function?

### One neuron, in code

Let's build a single neuron with 3 inputs. Notice that this is simply multivariate regression - supply a few variables and get the best-fit coefficients.

There are a few different [**activation** functions](https://en.wikipedia.org/wiki/Activation_function) beyond sigmoid, but importantly, they're all nonlinear continuous, differentiable functions. For now, we'll stick with sigmoid.

In [ ]:
def neuron(x, w, b):
    return sigmoid(np.dot(w, x) + b)

x = np.array([1.0, 2.0, 3.0])
w = np.array([0.2, -0.5, 0.1])
b = 0.0

print(f"Input x      = {x}")
print(f"Weights w    = {w}")
print(f"Bias b       = {b}")
print(f"w·x + b      = {np.dot(w, x) + b:.4f}")
print(f"Neuron output = {neuron(x, w, b):.4f}")

### Stacking neurons: a layer

A **layer** is just several neurons processing the same inputs in parallel. If we pack all their weight vectors into a matrix $\mathbf{W}$ and their biases into a vector $\mathbf{b}$:

$$\mathbf{h} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})$$

where $\sigma$ is applied element-wise. This is literally just matrix–vector multiplication plus a nonlinearity.

For example, in an image recognition model, each neuron might handle an individual pixel in the image. If the image resolution is 128 pixels, maybe 128 neurons. Higher resolution images need more neurons in a layer.

As a chemistry example, if you approximate a molecule as a [graph](https://en.wikipedia.org/wiki/Graph_neural_network) you might have multiple neurons in a layer to represent properties input from the atoms or bonds.

In [ ]:
def layer(x, W, b):
    return sigmoid(W @ x + b)

# A layer with 4 neurons, each taking a 3-dimensional input
W1 = np.random.randn(4, 3) * 0.5
b1 = np.zeros(4)

h = layer(x, W1, b1)
print(f"Layer output (4 neurons): {h}")

Notice in the code above, we created that layer of 4 neurons with a set of ***random*** weights to start with `np.random.randn(4, 3)` - so each of the 4 neurons takes in a 3D input and we've randomized the weights in the 4x3 matrix.

Eventually, we'll *train* the layer and the neurons. But it's important to realize that you start with some randomness for the weights.

### Two layers = a (tiny) neural network

Stack layers by feeding the output of one into the next:

$$\mathbf{h} = \sigma(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1), \qquad \hat{y} = \sigma(\mathbf{W}_2 \mathbf{h} + \mathbf{b}_2)$$

The **depth** (number of layers) and **width** (neurons per layer) are architectural choices. A network with untrained random weights does nothing useful — it's the training process that gives it meaning.

Notice that the depth and width can vary. Going back to the image model, maybe you start with a neuron for every pixel. Then the next layer groups together 4 pixel neurons into a 2x2 input into the neurons. The third layer might group together a 2x2 input from the second layer and so on. One layer is getting pixel details, the next layer is pooling together larger and larger groups of pixels for information.

<img src="https://upload.wikimedia.org/wikipedia/commons/4/46/Colored_neural_network.svg" alt="schematic of a neural network with input layer, hidden layer, and output">

Image from [Wikipedia](https://en.wikipedia.org/wiki/Neural_network_(machine_learning))

One of the reasons that machine learning has really accelerated in recent years is that computer hardware has advanced that [**deep** neural networks](https://en.wikipedia.org/wiki/Deep_learning) are feasible, with millions, billions, or even trillions of weights.

**Conceptual Questions for Discussion**

1. Each layer computes a weighted sum of simple functions. When else have we seen something like this?
2. What does a *wide* layer get you in terms of approximating a complicated function?
3. What do you think a *deep* set of layers might get you?

---
## 2. Training is optimization

We have a network with parameters $\boldsymbol{\theta} = \{\mathbf{W}_1, \mathbf{b}_1, \mathbf{W}_2, \mathbf{b}_2, \ldots\}$. **Training** means: find values of $\boldsymbol{\theta}$ that make the network's predictions best fit the data.

We define a **loss function** $\mathcal{L}(\boldsymbol{\theta})$ that is small when predictions are good. For example, maybe we're trying to minimize the mean-absolute error (MAE) or mean absolute percent error (MAPE), etc. much like we minimize errors in linear regression. Choice of loss function is an important concern.

1. Pick your loss $\mathcal{L}(\boldsymbol{\theta})$ that's small when predictions are good.
2. Compute $\nabla L$ — the gradient, a vector of partial derivatives.
3. Take a small step *against* the gradient: $\mathbf{w} \leftarrow \mathbf{w} - \eta \, \nabla_\mathbf{w} L$ where $\eta$ is the **learning rate** (step size)
4. Repeat.

This is just gradient descent, conjugate gradients or other numerical optimization method (BFGS, etc.) that we discussed earlier.

> **🧪 Chemistry analogy.** This is the *same* algorithm you use for geometry optimization: minimize an energy surface $E(\mathbf{R})$ by stepping downhill along $-\nabla_{\mathbf{R}} E$. Here the "coordinates" are network weights and the "energy" is the loss. High-dimensional — a modern LLM has $\sim 10^{11}$ "atoms" — but conceptually identical.

### A chemistry-flavored dataset

Let's make the network learn something useful: **classify molecules as water-soluble or not**, based on two simple features:

- $x_1$: molecular weight (scaled)
- $x_2$: ratio of heteroatoms (N, O) to total heavy atoms

We'll generate a toy dataset that captures the real trend: small polar molecules dissolve; large nonpolar ones don't. (There's a lot more to solubility prediction, but it's obviously a **very** useful problem to solve with machine learning. "Hey Siri, can you recommend a solvent for this compound?")


In [ ]:
# Toy dataset: 200 "molecules"
N = 200
MW = np.random.uniform(30, 400, N)              # molecular weight (g/mol)
het_ratio = np.random.uniform(0, 0.6, N)        # fraction of heteroatoms

# Ground-truth rule: soluble if small AND polar-ish (with some noise)
logit = -0.02 * MW + 8 * het_ratio + 1.0 + np.random.randn(N) * 0.5
y_true = (logit > 0).astype(float)

# Feature matrix (scale MW so both features are order-1)
X = np.column_stack([MW / 100.0, het_ratio])

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[y_true==1, 0]*100, X[y_true==1, 1], c='steelblue', label='soluble', alpha=0.7)
ax.scatter(X[y_true==0, 0]*100, X[y_true==0, 1], c='tomato', label='insoluble', alpha=0.7)
ax.set_xlabel('Molecular weight (g/mol)')
ax.set_ylabel('Heteroatom ratio')
ax.set_title('Toy solubility dataset')
ax.legend(); ax.grid(alpha=0.3); plt.show()

### Warm-up: one neuron = logistic regression

For a single neuron with sigmoid output predicting a 0/1 label, a natural loss is the **binary cross-entropy**:

$$\mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{N}\sum_{n=1}^{N} \Big[ y_n \log \hat{y}_n + (1 - y_n) \log(1 - \hat{y}_n) \Big]$$

This is what logistic regression optimizes. The beautiful fact is that for a sigmoid + cross-entropy combination, the gradient collapses to a remarkably simple form:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{1}{N}\sum_n (\hat{y}_n - y_n)\,\mathbf{x}_n, \qquad \frac{\partial \mathcal{L}}{\partial b} = \frac{1}{N}\sum_n (\hat{y}_n - y_n)$$

(derivation: chain rule + $\sigma'(z) = \sigma(z)(1 - \sigma(z))$. Try it.)


In [ ]:
# Logistic regression via gradient descent
w = np.zeros(2)
b = 0.0
lr = 0.5
losses = []

for step in range(500):
    y_pred = sigmoid(X @ w + b)
    eps = 1e-9  # numerical stability
    loss = -np.mean(y_true*np.log(y_pred+eps) + (1-y_true)*np.log(1-y_pred+eps))
    losses.append(loss)

    # gradients
    err = y_pred - y_true
    grad_w = X.T @ err / N
    grad_b = np.mean(err)

    # update
    w -= lr * grad_w
    b -= lr * grad_b

print(f"Final loss: {losses[-1]:.4f}")
print(f"Learned weights: {w}, bias: {b:.3f}")
print(f"Training accuracy: {((y_pred > 0.5) == y_true).mean():.3f}")

plt.plot(losses)
plt.xlabel('gradient descent step'); plt.ylabel('loss')
plt.title('Training a single neuron'); plt.grid(alpha=0.3); plt.show()

### Visualizing what one neuron learned

The decision boundary for a single sigmoid neuron is always a **straight line** — specifically, the line where $\mathbf{w}\cdot\mathbf{x} + b = 0$. The weights $\mathbf{w}$ set its orientation; the bias $b$ shifts it. Compare this to the curved boundary we'll get from the 2-layer network — that's what the hidden layer buys you.

In [ ]:
%pip install ipywidgets --user
from ipywidgets import interact, FloatSlider

def plot_single_neuron(w1, w2, bias):
    w_try = np.array([w1, w2])
    p_try = sigmoid(grid @ w_try + bias).reshape(xx.shape)
    acc = ((sigmoid(X @ w_try + bias) > 0.5) == y_true).mean()

    fig, ax = plt.subplots(figsize=(7, 5))
    cs = ax.contourf(xx*100, yy, p_try, levels=20, cmap='RdBu', alpha=0.6)
    plt.colorbar(cs, ax=ax, label='P(soluble)')
    ax.scatter(X[y_true==1, 0]*100, X[y_true==1, 1],
               c='steelblue', edgecolor='k', label='soluble')
    ax.scatter(X[y_true==0, 0]*100, X[y_true==0, 1],
               c='tomato', edgecolor='k', label='insoluble')
    ax.set_xlabel('Molecular weight (g/mol)')
    ax.set_ylabel('Heteroatom ratio')
    ax.set_title(f'Your accuracy: {acc:.3f}   '
                 f'(gradient descent got {0.905:.3f} — can you beat it?)')
    ax.legend(loc='upper right')
    plt.show()

interact(plot_single_neuron,
         w1=FloatSlider(value=0.0, min=-6, max=6,  step=0.1, description='w₁'),
         w2=FloatSlider(value=0.0, min=-6, max=10, step=0.1, description='w₂'),
         bias=FloatSlider(value=0.0, min=-6, max=6, step=0.1, description='b'));

### Scaling up: a 2-layer network with backpropagation

For networks with hidden layers, we use the chain rule systematically — this is [**backpropagation**](https://en.wikipedia.org/wiki/Backpropagation). The math is just the chain rule applied layer by layer; the code below implements it directly.

We won't derive it on the board today, but the key insight is: **gradients flow backwards through the network, layer by layer, reusing intermediate quantities from the forward pass**. In modern ML frameworks, this is handled through careful bookkeeping of [**automatic differentiation**](https://en.wikipedia.org/wiki/Automatic_differentiation).

In [ ]:
# 2-layer network: 2 inputs -> 8 hidden -> 1 output
H = 8
W1 = np.random.randn(H, 2) * 0.5;  b1 = np.zeros(H)
W2 = np.random.randn(1, H) * 0.5;  b2 = np.zeros(1)

lr = 0.3
losses = []

for step in range(3000):
    # ---- forward pass ----
    z1 = X @ W1.T + b1              # (N, H)
    h  = sigmoid(z1)                # (N, H)
    z2 = h @ W2.T + b2              # (N, 1)
    y_pred = sigmoid(z2).ravel()    # (N,)

    eps = 1e-9
    loss = -np.mean(y_true*np.log(y_pred+eps) + (1-y_true)*np.log(1-y_pred+eps))
    losses.append(loss)

    # ---- backward pass (chain rule) ----
    dL_dz2 = (y_pred - y_true).reshape(-1, 1) / N      # (N, 1)
    dL_dW2 = dL_dz2.T @ h                              # (1, H)
    dL_db2 = dL_dz2.sum(axis=0)                        # (1,)

    dL_dh  = dL_dz2 @ W2                               # (N, H)
    dL_dz1 = dL_dh * h * (1 - h)                       # sigmoid'
    dL_dW1 = dL_dz1.T @ X                              # (H, 2)
    dL_db1 = dL_dz1.sum(axis=0)                        # (H,)

    # ---- update ----
    W2 -= lr * dL_dW2; b2 -= lr * dL_db2
    W1 -= lr * dL_dW1; b1 -= lr * dL_db1

print(f"Final loss: {losses[-1]:.4f}")
print(f"Training accuracy: {((y_pred > 0.5) == y_true).mean():.3f}")

plt.plot(losses)
plt.xlabel('gradient descent step'); plt.ylabel('loss')
plt.title('Training a 2-layer network'); plt.grid(alpha=0.3); plt.show()

## A few notes

This is really a toy model. In reality, the loss / error doesn't always decrease quite so smoothly because we're trying to fit some complicated unknown function that probably has lots of noise or variation. (Solubility, for example, is a combination of many different factors of both solvent and solute.)

**Conceptual Questions for Discussion**
1. The loss went down during training. Is it guaranteed to keep going down if we train longer? How do we know when to stop training?
2. We trained on 200 molecules. If I showed the network a brand-new molecule, should I trust its prediction? What would I want to check first?
3. How might I prevent overfitting? (There are multiple strategies.)

### Visualizing what the network learned

Let's plot the network's decision boundary — the curve where it is 50% unsure.

This is actually a useful feature of neural networks - you can get some *uncertainty* in the predictions.

In [ ]:
xx, yy = np.meshgrid(np.linspace(0, 4.5, 100), np.linspace(0, 0.65, 100))
grid = np.column_stack([xx.ravel(), yy.ravel()])

h_grid = sigmoid(grid @ W1.T + b1)
p_grid = sigmoid(h_grid @ W2.T + b2).reshape(xx.shape)

plt.contourf(xx*100, yy, p_grid, levels=20, cmap='RdBu', alpha=0.6)
plt.colorbar(label='P(soluble)')
plt.scatter(X[y_true==1, 0]*100, X[y_true==1, 1], c='steelblue', edgecolor='k', label='soluble')
plt.scatter(X[y_true==0, 0]*100, X[y_true==0, 1], c='tomato', edgecolor='k', label='insoluble')
plt.xlabel('Molecular weight (g/mol)'); plt.ylabel('Heteroatom ratio')
plt.title('Learned decision boundary'); plt.legend(); plt.show()

> **🎯 Try this.** Change `H` (hidden layer width) to 2, or 50. Change `lr`. Change the number of training steps. What breaks? This intuition — that training is finicky and depends on hyperparameters — is real and scales up to billion-parameter models.


---
## Part 3 — From numbers to language

Everything so far: numbers in, numbers out. That's often useful enough for chemistry. Take in some properties (e.g., atomic partial charges, bond orders, etc.) and predict something (e.g., molecular solubility, pKa, etc.)

To do language, we need three ideas:

1. **Tokenization** — turn text into a sequence of integers.
2. **Embeddings** — turn each integer into a vector (so the network can do math on it).
3. **Next-token prediction** — the model outputs a probability distribution over the next token (or batch of tokens).

### 3.1 Tokenization

Words are too coarse (millions of them, many rare) and characters are too fine (long sequences). Modern LLMs use **subword tokens** — frequent chunks of text. The mapping is fixed and learned once from a big corpus.

The downside is that asking questions about characters can lead to weird results. "How many R's in blueberry?" is surprisingly hard for LLMs to answer.

In [ ]:
# tiktoken is the tokenizer used by GPT-4; no model weights needed.
%pip install tiktoken --user
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # GPT-4's tokenizer

examples = [
    "Hello, world!",
    "The Schrödinger equation describes quantum systems.",
    "acetylsalicylic acid",
    "CH3CH2OH",
]

for s in examples:
    ids = enc.encode(s)
    pieces = [enc.decode([i]) for i in ids]
    print(f"{s!r}")
    print(f"  -> {ids}")
    print(f"  -> {pieces}\n")

Notice a few things:

- Common words get a single token; rarer chemistry terms get split.
- Whitespace is usually attached to the following word.
- The vocabulary has ~100,000 tokens in GPT-4's tokenizer.

### 3.2 Embeddings: tokens → vectors

A tokenizer gives you integers. But the integers themselves have no meaningful arithmetic — token 523 isn't "half" of token 1046. So we associate each token with a learned vector in $\mathbb{R}^d$ (for GPT-3, $d = 12288$). This lookup is just a big matrix $\mathbf{E} \in \mathbb{R}^{V \times d}$ where row $i$ is the embedding of token $i$.

After training, the geometry of this embedding space becomes meaningful: similar words end up near each other. "Close" can be measured by cosine similarity — which is just a normalized dot product:

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$

Let's demo this with hand-placed 2D vectors — a caricature, but the geometry is real.

In chemistry applications, you might have similar situations for **embedding** category features, e.g. you want to supply different elements, but the atomic numbers aren't meaningful arithmetic. Carbon is not somehow six times a hydrogen atom even though it's six protons for both. Solvents can't be represented by a single number either.

In [ ]:
# Illustrative 2D embeddings (not trained — just to show the concept)
# Imagine these are the first 2 principal components of real learned embeddings
toy_embeddings = {
    # chemistry
    'methanol':  ( 2.8,  1.2),
    'ethanol':   ( 2.9,  1.4),
    'propanol':  ( 3.0,  1.5),
    'water':     ( 2.5,  0.8),
    'benzene':   ( 1.8, -1.2),
    'toluene':   ( 1.9, -1.0),
    'hexane':    ( 2.2, -1.5),
    # animals
    'dog':       (-2.5,  0.9),
    'cat':       (-2.4,  1.1),
    'puppy':     (-2.7,  0.6),
    'kitten':    (-2.6,  0.8),
    # verbs
    'run':       (-0.3, -2.5),
    'walk':      (-0.2, -2.4),
    'sprint':    (-0.5, -2.7),
}

fig, ax = plt.subplots(figsize=(8, 6))
for word, (x_, y_) in toy_embeddings.items():
    ax.scatter(x_, y_, s=50, c='steelblue')
    ax.annotate(word, (x_, y_), xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('embedding dim 1'); ax.set_ylabel('embedding dim 2')
ax.set_title('Semantic clustering in embedding space (illustrative)')
ax.grid(alpha=0.3); ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
plt.show()

Real learned embeddings show this structure — and much more. The famous example: $\text{vec}(\text{king}) - \text{vec}(\text{man}) + \text{vec}(\text{woman}) \approx \text{vec}(\text{queen})$. Analogies become vector arithmetic.

Obviously, a good embedding encodes the complex similarities in real language. "Rock" is similar to "mineral" but also somehow "music."

### 3.3 A language model in NumPy: the bigram

The simplest possible language model: estimate $P(w_{t+1} \mid w_t)$ — the probability of each word given only the previous word. We count occurrences in a corpus and normalize.

This *is* a language model. It's a bad one, but it has all the structural ingredients of GPT:
- a vocabulary
- a function from context to a probability distribution over next tokens
- sampling to generate text


In [ ]:
# Tiny toy corpus (write your own in class — students love this)
corpus = '''
the alkane reacts with the halogen to form a halogenated product .
the alcohol reacts with the acid to form an ester and water .
the acid reacts with the base to form a salt and water .
the aldehyde oxidizes to form a carboxylic acid .
the ketone does not oxidize easily under mild conditions .
the ester hydrolyzes to form an alcohol and an acid .
'''

tokens = corpus.split()
vocab = sorted(set(tokens))
V = len(vocab)
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
print(f"Vocabulary size: {V}")
print(f"Vocabulary: {vocab}")

# Count bigrams
counts = np.zeros((V, V))
for a, b in zip(tokens, tokens[1:]):
    counts[stoi[a], stoi[b]] += 1

# Normalize each row to a probability distribution over next words
# (add a small constant for stability — a 'smoothing' trick)
probs = (counts + 0.1) / (counts + 0.1).sum(axis=1, keepdims=True)

Now let's look at what the model predicts after the word `"reacts"`.

In [ ]:
context = 'reacts'
row = probs[stoi[context]]
order = np.argsort(-row)
print(f"Top predictions after '{context}':")
for i in order[:5]:
    print(f"  P({itos[i]!r} | {context!r}) = {row[i]:.3f}")

We now have a working text prediction model! Much more likely to get "reacts with" than other options. If we change the context to "ester", we can see "hydrolyzes" is a good option.

And the context doesn't need to be just one word. We can improve the model to have a larger "context window" to make better predictions - or predict more than one word at once. Of course $31^{10}$ is kind of a big number, so compressing the context is important.

### 3.4 Sampling and temperature

Given a probability distribution over the next token, how do we *generate* text? We sample. The **temperature** $T$ controls how adventurous the sampling is. Given logits $\mathbf{z}$, the distribution is

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- $T \to 0$: greedy — always pick the most likely token (deterministic, repetitive).
- $T = 1$: sample from the raw distribution.
- $T \to \infty$: uniform — pick randomly (gibberish).

This is *exactly* the temperature slider in ChatGPT / Claude API calls. We don't usually see this when we use a website, but it's in there. Obviously the extreme values don't work well - so when you're developing a model, you want to find a "good temperature" that provides some interesting variation. But it also means there's some randomness inherent in the output.


In [ ]:
def generate(start, n=12, temperature=1.0):
    out = [start]
    current = start
    for _ in range(n):
        row = probs[stoi[current]]
        # apply temperature
        logits = np.log(row + 1e-12)
        scaled = logits / temperature
        p = np.exp(scaled - scaled.max())
        p /= p.sum()
        nxt = np.random.choice(V, p=p)
        out.append(itos[nxt])
        current = itos[nxt]
    return ' '.join(out)

np.random.seed(0)
print("T = 0.3 (conservative):")
print(" ", generate('the', n=15, temperature=0.3))
print("\nT = 1.0 (normal):")
print(" ", generate('the', n=15, temperature=1.0))
print("\nT = 5.0 (chaotic):")
print(" ", generate('the', n=15, temperature=5.0))

### 3.5 Why this isn't enough — and what GPT does differently

Our bigram model has obvious problems:

- **No long-range context.** After seeing `"the alcohol reacts with the acid to form"`, our model only uses `"form"` to predict the next word.
- **Explicit count table.** The model is literally a $V \times V$ matrix. Scaling to a context of 20 words would need a $V^{20}$ table — impossible, much less 100,000 or more tokens. (And consider adapting to other languages!)

GPT replaces the count table with a **neural network**:
- Input: the last $k$ token embeddings (context window).
- Architecture: a stack of [**transformer blocks**](https://en.wikipedia.org/wiki/Transformer_(deep_learning)), each containing [**attention**](https://en.wikipedia.org/wiki/Attention_Is_All_You_Need) (which lets every token look at every other token) and an MLP (the same kind we built in Part 2).
- Output: a probability distribution over the next token — exactly like our bigram, just produced by a ~$10^{11}$-parameter neural network instead of a lookup.

> **Attention in one sentence:** at each layer, every token computes a weighted average of all the other tokens in the context, where the weights ("attention scores") are themselves computed from the tokens — so the model learns *what to look at* (e.g., important words / tokens).

Training is identical in spirit to Part 2: predict the next token, compute cross-entropy loss, backpropagate, update weights. Scaled up to trillions of tokens of text.

If you're curious, a variety of these models are available as open source, and you can use the tokenizer and embeddings, e.g. `from transformers import GPT2Tokenizer, GPT2Model` in the [`transformers`](https://huggingface.co/transformers) package.

**Conceptual Questions**

1. In the embedding demo, similar words clustered together. How do you decide where they should go?
2. We sampled from a probability distribution to generate text. But when you ask the same question twice, you often get similar answers. Why isn't it completely different each time?
3. Why do you think "hallucinations" show up in generated responses?

---
## Part 4 — Scaling, limits, and chemistry connections

### Why did this work so well?

The surprising empirical finding of the past decade: **the same architecture, trained on more data with more parameters, usually keeps getting better** — in smooth, predictable ways ("scaling laws"). Loss on the next-token prediction task decreases as a power law in compute, data, and parameters. And at some point, models that were trained *only* to predict the next word in text started exhibiting behaviors we call reasoning, coding, and translation — capabilities that emerge from the pressure to compress language well.

### How they can improve?

- **Beyond the base training - reinforcement learning** Some training is obvious. Get as much text as possible. Get as many books, newspapers, textbooks as possible. Read the internet. (Maybe useful, maybe problematic?) What do you do next? Generate some output and score it. This works *really* well with code because you can check if the Python syntax is correct and runs. Much harder to decide if something is *true*.
- **System prompts** Part of the context provided to the model is a "system prompt" that provides key information, e.g. "if the user is seeking medical advice, refer them to a doctor or medical professional." Surprisingly many new features can come through prompting, e.g "You are a Python tutor for Chem 1000 students, who are new to programming. Don't directly provide them with answers, only hints." The system prompts for [Claude are published](https://platform.claude.com/docs/en/release-notes/system-prompts).
- **Thinking context** Many current models generate an internal "thinking" context before producing an output. These ["reasoning models"](https://en.wikipedia.org/wiki/Reasoning_model) are intended to create a "chain of thought" that obviously influences the final output. This obviously helps in some kinds of tasks like programming or math problems.
- **Tool use** Many current models also now have the capability to use specific tools (e.g., "search the web" or "run this Python script"). This also helps with some tasks, since the LLM can generate code, run it, and get the output (e.g., drawing molecules).

### What this does not mean

- **LLMs don't have a world model in the way you or I do.** They have a model of *text about the world*, which is correlated with the world but not identical to it.
- **They hallucinate.** The training objective is "assign high probability to plausible text" — not "say true things". A fluent-sounding wrong answer and a fluent-sounding right answer have the same loss.
- **No built-in arithmetic.** They do arithmetic by pattern-matching on training data, not by running the algorithms you learned in grade school. This gets better with tools (code execution) because they now can create Python code and run it.

### Chemistry connections

The same machinery is being applied to chemistry in several ways. A few examples worth knowing about:

- **SMILES-based language models** (ChemBERTa, MolGPT): treat molecules as strings and apply the LLM recipe directly — pretraining, fine-tuning, generation.
- **Graph neural networks** for molecular property prediction: similar math, but the "tokens" are atoms and the "context" is the molecular graph.
- **Equivariant neural network potentials** (NequIP, MACE, Allegro): learned force fields that respect physical symmetries. These are starting to rival DFT accuracy at classical-force-field cost.
- **Retrosynthesis and reaction prediction**: sequence-to-sequence models trained on reaction databases.

The math you've been learning this semester — linear algebra, multivariable calculus, optimization — is the same math that makes all of this work. You already have the tools to read papers in this space.

---

**Conceptual Discussion Questions**

1. I claimed a language model is "a function that returns a probability distribution over the next word." If that's all it is, why does ChatGPT seem to answer questions, reason, translate, write code...?
2. An LLM trained on organic chemistry textbooks can probably answer most undergraduate exam questions correctly. Does that mean it "understands" organic chemistry? What would count as evidence either way?
3. Why can a model like ChatGPT or Claude give fluent but factually wrong chemistry answers? Where in the pipeline does "truth" live — or not live?

## Take-home exercises

1. **Larger vocabulary.** Paste a longer chemistry paragraph into `corpus` and retrain the bigram model. How does the quality change?
2. **Trigram model.** Extend the bigram to a trigram: $P(w_{t+1} \mid w_{t-1}, w_t)$. What's the memory cost? Connect this to why we need neural networks for real language modeling.
3. **Learn SMILES.** A common string representation for molecular chemistry is [SMILES: Simplified Molecular Input Line Entry System](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System). Write out some molecules in SMILES. How might you break molecules into tokens?
4. **Activation Functions.** One of the most common activation functions is [ReLU](https://en.wikipedia.org/wiki/Rectified_linear_unit). Consider why it might be more useful for large ML models.

-------
This notebook is from Prof. Geoffrey Hutchison, University of Pittsburgh
https://github.com/ghutchis/chem1000

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a>